# RSNA Knee Abnormality Detection — Submission Notebook

Bu notebook Kaggle Code Competition kurallarına uygun şekilde yazılmıştır:
- **Internet: Off** olmalı (Settings panelinden kapatılmalı)
- Eğitilmiş model ağırlıkları ayrı bir Kaggle Dataset olarak eklenmeli ve `MODEL_DIR` güncellenmeli
- Çalışma süresi 9 saati aşmamalı
- Çıktı dosyası `submission.csv` olmalı

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch

sys.path.append('/kaggle/input/rsna-knee-src/src')  # dataset olarak eklenen src/ klasoru
from dataset import KneeMRIDataset, LABEL_COLS
from model import KneeMultiViewModel

DATA_DIR = '/kaggle/input/rsna-knee-abnormality-detection'
MODEL_DIR = '/kaggle/input/rsna-knee-model-weights'  # kendi model dataset adınla degistir
N_FOLDS = 5
TARGET_SLICES = 24
RESIZE_TO = 224
BATCH_SIZE = 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
test_series_df = pd.read_csv(f'{DATA_DIR}/test_series.csv')
print(test_df.shape, test_series_df.shape)
test_df.head()

In [ ]:
test_ds = KneeMRIDataset(
    labels_df=test_df,
    series_df=test_series_df,
    series_root=f'{DATA_DIR}/test_series',
    target_slices=TARGET_SLICES,
    resize_to=RESIZE_TO,
    is_test=True,
)
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

In [ ]:
# K-fold modellerini yukle, tahminleri ortala (ensembling)
models = []
for fold in range(N_FOLDS):
    ckpt_path = f'{MODEL_DIR}/model_fold{fold}.pt'
    if not os.path.exists(ckpt_path):
        continue
    m = KneeMultiViewModel(pretrained=False).to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device))
    m.eval()
    models.append(m)
print(f'{len(models)} fold modeli yuklendi')

In [ ]:
all_study_uids = []
all_probs = []

with torch.no_grad():
    for views, study_uids in test_loader:
        views = views.to(device)
        fold_probs = []
        for m in models:
            logits = m(views)
            fold_probs.append(torch.sigmoid(logits).cpu().numpy())
        avg_probs = np.mean(fold_probs, axis=0) if fold_probs else np.full(
            (views.shape[0], len(LABEL_COLS)), 0.5
        )
        all_probs.append(avg_probs)
        all_study_uids.extend(study_uids)

all_probs = np.concatenate(all_probs, axis=0)
print(all_probs.shape)

In [ ]:
submission = pd.DataFrame(all_probs, columns=LABEL_COLS)
submission.insert(0, 'StudyInstanceUID', all_study_uids)

# Guvenlik: test.csv'deki tum studyler kapsanmis mi kontrol et
assert set(submission['StudyInstanceUID']) == set(test_df['StudyInstanceUID']), \
    'Eksik study var, submission tamamlanmadan kontrol et!'

submission.to_csv('submission.csv', index=False)
submission.head()